<a href="https://colab.research.google.com/github/oooinr4018-web/-1/blob/main/ESAA_1002_%EA%B3%BC%EC%A0%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#7. 행렬 분해를 이용한 잠재 요인 협업 필터링 실습

행렬 분해 잠재 요인 협업 필터링: SVD, NMF 등 적용

일반적인 행렬 분해: SVD

사용자-아이템 평점 행렬: SGD, ALS 기반 행렬 분해

< SGD 기반 행렬 분해> -> 영화 추천

4절 잠재 요인 협업 필터링 절의 확률적 경사 하강법을 이용한 행렬 분해에서 사용한 예제 코드 활용

- 행렬 분해 로직 부분을 함수로 생성

- 새로운 주피터 노트북 생성

- 확률적 경사 하강법을 이용한 행렬 분해 예제의 get_rmse() 함수 사용

- 행렬 분해 로직을 새로운 matrix_factorization() 함수로 정리

matrix_factorization(R, K, steps=200, learning_rate=0.01, r_lambda=0.01)

: R - 원본 사용자-아이템 평점 행렬 / K - 잠재 요인의 차원 수 / steps - SGD의 반복 횟수 / learning_rate - 학습률 / r_lambda - L2 규제 계수



In [2]:
import numpy as np

def get_rmse(R, P, Q, non_zeros):
  error = 0

  for i, j, r in non_zeros:
    pred = np.dot(P[i, :], Q[j, :].T)
    error += pow(r - pred, 2)
    rmse = np.sqrt(error / len(non_zeros))

    return rmse

In [5]:
def matrix_Factorization(R, K, steps=200, learning_rate=0.01, r_lambda=0.01):
  num_users, num_iters=R.shape
  # P와 Q 매트릭스의 크기를 지정하고 정규 분포를 가진 랜덤한 값으로 입력합니다.
  np.random.seed(1)
  P=np.random.normal(scale=1./K, size=(num_users, K))
  Q=np.random.normal(scale=1./K, size=(num_iters, K))

  # R>0인 행 위치, 열 위치, 값을 num_zeros 리스트 객체에 저장.
  non_zeros=[ (i, j, R[i, j]) for i in range(num_users) for j in range(num_iters) if R[i, j]>0 ]

  # SGD 기법으로 P와 Q 매트릭스를 계속 업데이트.
  for step in range(steps):
    for i, j, r in non_zeros:
      # 실제 값과 예측 값의 차이인 오류 값 구함
      eij=r-np.dot(P[i,:]. Q[j,:].T)
      # Regularization을 반영한 SGD 업데이트 공식 적용
      P[i, :]=P[i, :]+learning_rate*(eij*Q[j,:]-r_lambda*P[i,:])
      Q[j, :]=Q[j, :]+learning_rate*(eij*P[i,:]-r_lambda*Q[j,:])

    rmse=get_rmse(R, P, Q, non_zeros)
    if (step%10)==0:
      print("### iteration step :", step, "rmse :", rmse)

  return P, Q

In [15]:
!wget https://files.grouplens.org/datasets/movielens/ml-latest-small.zip
!unzip ml-latest-small.zip

--2026-10-02 04:32:50--  https://files.grouplens.org/datasets/movielens/ml-latest-small.zip
Resolving files.grouplens.org (files.grouplens.org)... 128.101.96.204
Connecting to files.grouplens.org (files.grouplens.org)|128.101.96.204|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 978202 (955K) [application/zip]
Saving to: ‘ml-latest-small.zip.1’

ml-latest-small.zip 100%[===================>] 955.28K  3.21MB/s    in 0.3s    

2026-10-02 04:32:50 (3.21 MB/s) - ‘ml-latest-small.zip.1’ saved [978202/978202]

Archive:  ml-latest-small.zip
replace ml-latest-small/links.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [17]:
# 영화 평점 행렬 데이터를 새롭게 DataFrame으로 로딩 -> 사용자-아이템 평점 행렬로 생성

import pandas as pd
import numpy as np

movies=pd.read_csv('./ml-latest-small/movies.csv')
ratings=pd.read_csv('./ml-latest-small/ratings.csv')
ratings=ratings[['userId', 'movieId', 'rating']]
ratings_matrix=ratings.pivot_table('rating', index='userId', columns='movieId')

# title 칼럼을 얻기 위해 movies와 조인 수행
rating_movies=pd.merge(ratings, movies, on='movieId')

# columns='title'로 title 칼럼으로 pivot 수행.
ratings_matrix=rating_movies.pivot_table('rating', index='userId', columns='title')

In [19]:
def matrix_factorization(R, K, steps=200, learning_rate=0.01, r_lambda=0.01):
    num_users, num_items = R.shape

    np.random.seed(1)
    P = np.random.normal(scale=1./K, size=(num_users, K))
    Q = np.random.normal(scale=1./K, size=(num_items, K))

    non_zeros = [
        (i, j, R[i, j])
        for i in range(num_users)
        for j in range(num_items)
        if not np.isnan(R[i, j])
    ]

    for step in range(steps):
        for i, j, r in non_zeros:
            eij = r - np.dot(P[i, :], Q[j, :].T)

            P[i, :] += learning_rate * (
                eij * Q[j, :] - r_lambda * P[i, :]
            )
            Q[j, :] += learning_rate * (
                eij * P[i, :] - r_lambda * Q[j, :]
            )

    return P, Q

In [20]:
# 만들어진 사용자-아이템 평점 행렬을 matrix_factorization() 함수를 이용하여 행렬 분해
# SGD 반복 횟수인 steps=200회만 지정
# 잠재 요인 차원 K=50, 학습률=0.01, Regularization 계수=0.01

P, Q=matrix_factorization(ratings_matrix.values, K=50, steps=200, learning_rate=0.01,
                          r_lambda=0.01)
pred_matrix=np.dot(P, Q.T)

In [22]:
# 예측 사용자-아이템 평점 행렬을 영화 타이틀을 칼럼명으로 가지는 DataFrame으로 변경

ratings_pred_matrix=pd.DataFrame(data=pred_matrix, index=ratings_matrix.index,
                                 columns=ratings_matrix.columns)
ratings_pred_matrix.head(3)

title,'71 (2014),'Hellboy': The Seeds of Creation (2004),'Round Midnight (1986),'Salem's Lot (2004),'Til There Was You (1997),'Tis the Season for Love (2015),"'burbs, The (1989)",'night Mother (1986),(500) Days of Summer (2009),*batteries not included (1987),...,Zulu (2013),[REC] (2007),[REC]² (2009),[REC]³ 3 Génesis (2012),anohana: The Flower We Saw That Day - The Movie (2013),eXistenZ (1999),xXx (2002),xXx: State of the Union (2005),¡Three Amigos! (1986),À nous la liberté (Freedom for Us) (1931)
userId,,,,,,,,,,,,,,,,,,,,,
1,3.055084,4.092018,3.564130,4.502167,3.981215,1.271694,3.603274,2.333266,5.091749,3.972454,...,1.402608,4.208382,3.705957,2.720514,2.787331,3.475076,3.253458,2.161087,4.010495,0.859474
2,3.170119,3.657992,3.308707,4.166521,4.311890,1.275469,4.237972,1.900366,3.392859,3.647421,...,0.973811,3.528264,3.361532,2.672535,2.404456,4.232789,2.911602,1.634576,4.135735,0.725684
3,2.307073,1.658853,1.443538,2.208859,2.229486,0.780760,1.997043,0.924908,2.970700,2.551446,...,0.520354,1.709494,2.281596,1.782833,1.635173,1.323276,2.887580,1.042618,2.293890,0.396941


In [23]:
# 사용자 9번 영화추천
def get_unseen_movies(ratings_matrix, userId):
  # userId로 입력받은 사용자의 모든 영화 정보를 추출해 Series로 반환함.
  # 반환된 user_rating은 영화명(title)을 인덱스를 가지는 Series 객체임.
  user_rating = ratings_matrix.loc[userId, :]
  #user_rating이 0보다 크면 기존에 관람한 영화임. 대상 인덱스를 추출해 list 객체로 만듦.
  already_seen = user_rating[ user_rating > 0].index.tolist()
  # 모든 영화명을 list 객체로 만듦.
  movies_list = ratings_matrix.columns.tolist()
  # list comporehension으로 already_seen에 해당하는 영화는 movies_list에서 제외함.
  unseen_list = [ movie for movie in movies_list if movie not in already_seen]
  return unseen_list


def recomm_movie_by_userid(pred_df, userId, unseen_list, top_n = 10):
  # 예측 평점 DataFrame에서 사용자id 인덱스와 unseen_list로 들어온 영화명 칼럼을 추출해
  # 가장 예측 평점이 높은 순으로 정렬함.
  recomm_movies = pred_df.loc[userId, unseen_list].sort_values(ascending=False)[:top_n]

  return recomm_movies

In [24]:
# 예측 사용자-아이템 평점 행렬 정보를 이용하여 개인화된 영화 추천 진행
# 사용자 아이디 9번에 대한 영화 추천을 잠재 요인 협업 필터링으로 추천
# get_unseen_moveis() 함수, recomm_movie_by_userid() 함수로 추천 영화 추출

# 사용자가 관람하지 않은 영화명 추출
unseen_list=get_unseen_movies(ratings_matrix, 9)

# 잠재 요인 협업 필터링으로 영화 추천
recomm_movies=recomm_movie_by_userid(ratings_pred_matrix, 9, unseen_list, top_n=10)

# 평점 데이터를 DataFrame으로 생성.
recomm_movies=pd.DataFrame(data=recomm_movies.values, index=recomm_movies.index,
                           columns=['pred_score'])
recomm_movies

,pred_score
title,
Rear Window (1954),5.704612
"South Park: Bigger, Longer and Uncut (1999)",5.451100
Rounders (1998),5.298393
Blade Runner (1982),5.244951
Roger & Me (1989),5.191962
Gattaca (1997),5.183179
Ben-Hur (1959),5.130463
Rosencrantz and Guildenstern Are Dead (1990),5.087375
"Big Lebowski, The (1998)",5.038690


- 앞의 아이템 기반 협업 필터링 결과와 추천된 영화가 많이 다름.

- 알프레드 히치콕 감독의 스릴러 영화 '이창'이 추천됨.

- 어른을 위한 애니메이션 여화 '사우스파크'가 두 번째로, 영화 '라운더스', 그 뒤로 약간 어둡고 무거운 주제의 영화들('블레이드 러너', '로저와 나', '가타가')이 추천됨.

# 파이썬 추천 시스템 패키지 - Surprise

# Surprise 패키지 소개

앞 코드는 최적화, 수행 속도 측면에서의 보완이 필요

- 추천 시스템은 별도의 패키지로 제공되면 활용도가 매우 높음.

-> 추천 시스템 구축을 위한 전용 패키지 Surprise

- 파이썬 기반에서 사이킷런과 유사한 API, 프레임워크 제공

- conda나 pip을 통한 설치

In [26]:
! pip install scikit-surprise

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 36.8 MB/s eta 0:00:00


- surprise 패키지의 최신 버전은 1.1.1

- Surprise 패키지의 장점

1. 다양한 추천 알고리즘

(사용자 또는 아이템 기반 최근접 이웃 협업 필터링, SVD, SVS++, NMF 기반 잠재 요인 협업 필터링을 적용한 추천 시스템 구축)

2. Surprise의 핵심 API는 사이킷런의 핵심 API와 유사한 API

fit(), predict() API: 추천 데이터 학습, 예측

train_test_split(): 추천 학습 데이터 세트, 예측 데이터 세트 분리

cross_validate(), GridSearchCV 클래스 추천 시스템을 위한 모델 셀력션, 평가, 하이퍼 파라미터 튜닝 등의 기능 제공



# Surprise를 이용한 추천 시스템 구축

Surprise에 대한 문서: http://surprise.readthedocs.io/en/stable/에 정리되어있음.

예제를 통한 Surprise 패키지의 개략적인 사용법 익히기

-> 추천 데이터를 학습용, 테스트용 데이터 세트로 분리한 뒤, SVD 행렬 분해를 통한 잠재 요인 협업 필터링 수행

In [27]:
# 새로운 주피터 노트북 생성
# Surprise 관련 모듈 임포트

from surprise import SVD
from surprise import Dataset
from surprise import accuracy
from surprise.model_selection import train_test_split

- 데이터 세트 로딩 -> Surprise에서 데이터 로딩은 Dataset 클래스를 이용해서만 가능

- Surprise는 userID(사용자 ID), movieID(영화 ID), rating(평점)과 같은 주요 데이터가 Row 레벨 형태로 되어있는 포맷의 데이터만 처리

- Surprise는 무비렌즈(MovieLens) 사이트에서 제공하는 과거 버전의 데이터 세트를 가져오는 API 제공

Surprise Dataset 클래스의 load_builtin(): 무비렌즈 사이트에서 제공하는 과거 버전 데이터 세트인 'ml-100k', 'ml-1m' 데이터를 아카이브 사이트로부터 내려받아 로컬 디렉터리에 저장한 뒤 데이터 로딩

-> 로딩한 데이터 세트를 Surprise 패키지의 train_test_split() API를 이용하여 학습 데이터 세트 / 테스트 데이터 세트로 분리



In [29]:
data=Dataset.load_builtin('ml-100k')
# 수행 시마다 동일하게 데이터를 분할하기 위해 random_state 값 부여
trainset, testset= train_test_split(data, test_size=25, random_state=0)

- C:\Users\chkwon\.surprise_data\ml-100k 디렉터리에 데이터가 저장됨.

- 한 번 내려받아 로컬 디렉터리에 데이터가 저장 -> Dataset.load_builtin('m1-100k')을 호출하면 무비렌즈 사이트에 접속 X 저장된 데이터 세트 로딩

- Surprise에서 내려받은 ml-100k, ml-1m: 과거 버전의 데이터 세트

- ratings.csv: 칼럼 분리 문자가 콤마(,)인 CSV 파일 <-> 과거 버전에서는 분리 문자가 탭(\n) 문자

- Surprise에 사용자-아이템 평점 데이터 적용 시, 로우 레벨의 사용자-아이템 평점 데이터 그대로 적용 (무비렌즈 사이트에서 내려받은 데이터 파일과 동일)

(Surprise는 자체적으로 로우 레벨의 데이터를 칼럼 레벨의 데이터로 변경)

In [30]:
# SVD로 잠재 요인 협업 필터링 수행
# 적용 데이터 세트: train_test_split()으로 분리된 학습 데이터 세트
# algo=SVD()와 같은 알고리즘 객체 생성
# 알고리즘 객체에 fit(학습 데이터 세트) 수행 -> 학습 데이터 세트 기반 추천 알고리즘 학습

algo=SVD(random_state=0)
algo.fit(trainset)

- 학습된 추천 알고리즘 기반 테스트 데이터 세트에 대한 추천 수행

- Surprise에서 추천 예측 매서드: test(), predict()

test(): 사용자-아이템 평점 데이터 세트 전체에 대해 추천을 예측하는 매서드

(입력된 데이터 세트에 대한 추천 데이터 세트 생성)

predict(): 개별 사용자, 영화에 대한 추천 평점 반환

In [31]:
predictions=algo.test(testset)
print('prediction type:', type(predictions), 'size:', len(predictions))
print('prediction 결과의 최초 5개 추출')
predictions[:5]

prediction type: <class 'list'> size: 25
prediction 결과의 최초 5개 추출


[Prediction(uid='917', iid='50', r_ui=3.0, est=np.float64(4.149217910133892), details={'was_impossible': False}),
 Prediction(uid='235', iid='318', r_ui=5.0, est=np.float64(4.333029650171127), details={'was_impossible': False}),
 Prediction(uid='339', iid='12', r_ui=5.0, est=np.float64(4.652894591303144), details={'was_impossible': False}),
 Prediction(uid='878', iid='755', r_ui=2.0, est=np.float64(2.7545011408291202), details={'was_impossible': False}),
 Prediction(uid='618', iid='56', r_ui=4.0, est=np.float64(3.021383823173283), details={'was_impossible': False})]

- SVD 알고리즘 객체의 test(데이터 세트) 메서드의 호출 결과: 파이썬 리스트 (크기= 입력 데이터 세트의 크기 25000개)

- 호출 결과로 반환된 리스트 객체는 25000개의 Prediction 객체를 내부에 가지고 있음.

- Prediction 객체: Surprise 패키지에서 제공하는 데이터 타입

: 개별 사용자 아이디(uid), 영화(아이템) 아이디(iid), 실제 평점(r_ui) 정보에 기반한 Surprise 추천 예측 평점(est) 데이터를 튜플 형태로 가지고 있음.

: Prediction 객체의 details 속성은 내부 처리 시 추천 예측을 할 수 없는 경우 로그용으로 데이터를 남김.

- 'was_impossible'=True : 예측값 생성 X 데이터

- 객체명.uid와 같은 형식: 리스트 객체 내 Prediction 객체의 uid, iid, r_ui, est 등의 속성에 접근

In [32]:
# Prediction 객체에서 uid, iid, est 속성 추출 예제
[ (pred.uid, pred.iid, pred.est) for pred in predictions[:3]]

[('917', '50', np.float64(4.149217910133892)),
 ('235', '318', np.float64(4.333029650171127)),
 ('339', '12', np.float64(4.652894591303144))]

- Surprise 패키지의 추천 예측 메서드 predict()를 이용한 추천 예측

predict(): 개별 사용자의 아이템에 대한 추천 평점 예측

(인자: 개별 사용자 아이디, 아이템 아이디 -> 반환: 추천 예측 평점 포함한 정보)

(기존 평점 정보 (r_ui)는 선택 사항 / 사용자 아이디, 아이템 아이디는 문자열로 입력)

In [33]:
# 사용자 아이디, 아이템 아이디는 문자열로 입력해야 함
uid=str(196)
iid=str(302)
pred=algo.predict(uid, iid)
print(pred)

user: 196        item: 302        r_ui = None   est = 4.37   {'was_impossible': False}


- predict()에 개별 사용자, 아이템 정보 입력 -> 추천 예측 평점을 est로 반환

- test() 메서드: 입력 데이터 세트의 모든 사요자, 아이템 아이디에 대해 predict()를 반복 수행한 결과

- 테스트 데이터 세트를 이용한 추천 예측 평점, 실제 평점과의 차이 평가

Surprise의 accuracy 모듈: RMSE, MSE 등의 방법으로 추천 시스템의 성능 평가 정보 제공

accuracy 모듈의 rmse()를 이용한 RMSE 평가 결과 확인

In [34]:
accuracy.rmse(predictions)

RMSE: 0.9277


np.float64(0.9277328494227115)

# Surprise 주요 모듈 소개

내장 데이터 세트가 아닌 다른 데이터 파일을 로딩하여 추천 시스템 구현

이전 절의 ratings, csv, movies.csv 파일 이용한 실습

Dataset

- Surprise는 user_id(사용자 아이디), item_id(아이템 아이디), rating(평점) 데이터가 로우 레벨로 된 데이터 세트만 적용

데이터의 첫 번째 칼럼을 사용자 아이디, 두 번째 칼럼을 아이템 아이디, 세 번 칼럼을 평점, 네번째 칼럼부터 로딩 수행 X

- 무비렌즈 아카이브 서버에서 자동 다운 데이터 파일, 일반 데이터 파일, 판다스 DatFrame에서도 로딩 가능해야 함.

(데이터 세트의 칼럼 순서가 사용자 아이디, 아이템 아이디, 평점 순)

1. Dataset.load_builtin(name='ml-100k')

- 무비렌즈 아카이브 FTP 서버에서 무비렌즈 데이터 내려받기

내려받은 데이터는 .surprise_data 디렉터리 밑에 저장

(해당 디렉터리에 데이터가 있으면, FTP에서 내려받지 않고 해당 데이터 이용)

- 입력 파라미터 name으로 대상 데이터가 ml-100k인지, ml-1m인지 입력

(디폴트는 ml-100k)

2. Dataset.load_from_file(file_path, reader)

- OS 파일에서 데이터 로딩 시 사용

- 콤마, 탭 등으로 칼럼이 분리된 포맷의 OS 파일에서 데이터 로딩

- 입력 파라미터로 OS 파일명, Reader로 파일의 포맷 지정

3. Dataset.load_from_df(df, reader)

- 판다스의 DataFrame에서 데이터 로딩

- 파라미터는 DataFrame (DataFrame은 3개의 칼럼인 사용자 아이디, 아이템 아이디, 평점 순 칼럼 순서)

- 입력 파라미터로 DataFrame 객체, Reader로 파일의 포맷 지정

# OS 파일 데이터를 Surprise 데이터 세트로 로딩

Dataset.load_from_file API로, 지정된 디렉터리에 있는 사용자-아이템 평점 데이터 로딩

데이터 파일: ratngs.csv, movies.csv

- Surirse에 OS 파일 로딩 시 주의할 점: 로딩 데이터 파일에 칼럼명을 가지는 헤더 문자열 X

DatafFrame의 to_csv() 함수를 이용하여 칼럼 헤더 삭제 -> 새로운 파일 ratings_noh.csv로 저장

In [36]:
import pandas as pd

ratings=pd.read_csv('ratings.csv')
# ratings_noj.csv 파일로 업로드 시 인덱스와 헤더를 모두 제겅한 새로운 파일 생성.
ratings.to_csv('ratings_noh.csv', index=False, header=False)

- ratings_noh.csv: ratings_csv 파일에서 헤더가 삭제된 파일

- Reader 클래스를 이용하여 데이터 파일의 파싱 포맷 정의

(Reader 클래스의 생성자에 각 필드의 칼럼명, 칼럼 분리문자, 최소~최대 평점 입력하여 객체 생성, load_from_file()로 생성된 Reader 객체)

- ratings_noh.csv를 Dataset 모듈의 load_from_file()을 이용하여 Dataset으로 로드

<다음 예제>

Reader 설정

- line_format 인자로 user, item, rating, tumestamp의 4개의 칼럼으로 데이터가 구성되어 있음을 명시

- 각 칼럼의 분리 문자는 콤마(,), 평점 단위는 0.5, 최대 평점은 5로 설정

이후, 이를 기반하여 load_from_file()로 데이터 파싱 -> Dataset 로딩

(로딩 시, ratings_noh.csv 파일에서 앞 3개 칼럼만 로딩, timestamp 칼럼은 제외)

In [37]:
from surprise import Reader

reader=Reader(line_format='user item rating timestamp', sep=',', rating_scale=(0.5,5))
data=Dataset.load_from_file('ratings_noh.csv', reader=reader)

- Surprise 데이터 세트는 기본적으로 무비렌즈 데이터 형식을 따름.

- 데이터 세트가 다른 OS 파일의 경우, Reader 클래스 먼저 설정

Reader 클래스의 주요 생성 파라미터

line_format (string): 칼럼을 순서대로 나열

(입력된 문자열을 공백으로 분리하여 칼럼으로 인식)

sep(char): 칼럼을 분리하는 분리자

(디폴트 '\t', 판다스 DataFrame에서 입력받을 경우 기재 X)

rating_scale(tuple, optional): 평점 값의 최소~최대 평점 설정

(디폴트 (1,5), ratings_csv 파일의 최소 평점 0.5, 최대 평점 5 -> (0.5,5)로 설정)

- SVD 행렬 분해 기법을 이용한 추천 예측

- 잠재 요인 크기 K 값을 나타내는 파라미터 n_factors=50으로 설정

데이터 학습 -> 테스트 데이터 세트를 적용하여 예측 평점 구하기 -> 예측 평점, 실제 평점 데이터를 RMSE로 평가

In [38]:
trainset, testset=train_test_split(data, test_size=25, random_state=0)

# 수행 시마다 동일한 결과를 도출하기 위해 random_state 설정
algo=SVD(n_factors=50, random_state=0)

# 학습 데이터 세트로 학습하고 나서 테스트 데이터 세트로 평점 예측 후 RMSE 평가
algo.fit(trainset)
predictions=algo.test(testset)
accuracy.rmse(predictions)

RMSE: 0.7940


np.float64(0.7939608260279409)

- RMSE는 0.7940

# 판다스 DataFrame에서 Surprise 데이터 세트로 로딩

- Datset.load_from_df(): 판다스의 DataFrame에서도 Surprise 데이터 세트로 로딩 가능

(DataFrame에서 사용자 아이디, 아이템 아이디, 평점 칼럼 순서 지키기)

- Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']], reader): ratings.csv 파일을 DataFrame으로 로딩한 ratings에서 Surprise 데이터 세트로 로딩

In [40]:
# SVD 추천 예측 코드 작성

import pandas as pd
from surprise import Reader, Dataset

ratings=pd.read_csv('ratings.csv')
reader=Reader(rating_scale=(0.5, 5.0))

# ratings DataFrame에서 칼럼은 사용자 이읻, 아이템 아이디, 평점 순서를 지켜야 합니다.
data=Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']],reader)
trainset, testset=train_test_split(data, test_size=35, random_state=0)

algo=SVD(n_factors=50, random_state=0)
algo.fit(trainset)
predictions=algo.test(testset)
accuracy.rmse(predictions)

RMSE: 0.8794


np.float64(0.8794288205361218)

# Surprise 추천 알고리즘 클래스

1. SVD: 행렬 분해를 통한 잠재 요인 협업 필터링을 위한 SVD 알고리즘

2. KNNBasic: 최근접 이웃 협업 필터링을 위한 KNN 알고리즘

3. BaselineOnly: 사용자 Bias, 아이템 Bias를 감안한 SGD 베이스라인 알고리즘

이외의 SVD++, NMF 등 다양한 유형의 알고리즘 수행 가능

알고리즘은 surprise 사이트 문서에서 참조 가능

Surprise SVD의 비용 함수: 사용자 베이스라인 편향성을 감안한 평점 예측에 Regularization 적용

SVD 클래스의 입력 파라미터

1. n_factors: 잠재 요인 K 개수

(디폴트는 100, 커질수록 정확도가 높아짐, 과적합 가능성)

2. n_epochs: SGD 수행 시 반복 횟수

(디폴트는 20)

3. biased (bool): 베이스라인 사용자 편향 적용 여부

(디폴트는 True)

추천 알고리즘의 예측 성능 벤치마크 결과는 http://surpriselib.com/에서 확인 가능

*벤치마크: Core i5 7th gen (2.5 GHz), 8G RAM 상에서 100k 데이터 세트로 테스트한 결과

- SVD++ 알고리즘의 RMSE, MAE 성적이 가장 좋음. 상대적으로 시간이 너무 오래 걸림. -> 사용하기 수월하지 X

- SVD, k-NN Baseline의 성능 평가 수치가 좋음.

(k-NN 자체는 성능이 안좋지만, Baseline을 결합한 경우 성능 평가 수치 대폭 향상)

Baseline: 각 개인이 평점을 부여하는 성향을 반영하여 평점을 계산하는 방식

# Baseline 평점

-개인의 성향에 따라 영화나 상품의 평가도 같은 아이템이더라도 평가가 달라질 수 있음.

- 베이스라인 평점: 개인의 성향을 반영해 아이템 평가에 편향성 요소를 반영하여 평점을 부과하는 것

베이스라인 평점 = 전체 평균 평점 + 사용자 편향 점수 + 아이템 편향 점수

전체 평균 평점 = 모든 사용자의 아이템에 대한 평점을 평균한 값

사용자 편향 점수 = 사용자별 아이템 평점 평균 값 - 전체 평균 평점

아이템 편향 점수 = 아이템별 평점 평균 값 - 전체 평균 평점

# 교차 검증과 하이퍼 파라미터 튜닝

Surprise는 교차 검증, 하이퍼 파라미터 튜닝을 위해 cross_validate(), GridSearchCV 클래스 제공

1. cross_validate() 함수

- surprise.model_selection 모듈 내에 존재

- 폴드된 데이터 세트의 개수, 성능 측정 방법을 명시하여 교차 검증 수행

- cross_validate()의 인자: 알고리즘 객체, 데이터, 성능 평가 방법(measures), 폴드 데이터 세트 개수(cv)

<다음 예제>

cross_validate()를 이용하여 ratings.csv를 DataFrame으로 로딩한 데이터를 5개의 학습/검증 폴드 데이터 세트로 분리 -> 교차 검증 수행 -> RMSE, MAE로 성능 평가 진행





In [43]:
from surprise.model_selection import cross_validate

# 판다스 DataFrame에서 Surprise 데이터 세트로 데이터 로딩
ratings=pd.read_csv('ratings.csv') # reading data in pandas df
reader=Reader(rating_scale=(0.5, 5.0))
data=Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']], reader)

algo=SVD(random_state=0)
cross_validate(algo, data, measures=['RMSE', 'MAE'], cv=5, verbose=True)

Evaluating RMSE, MAE of algorithm SVD on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    0.8688  0.8809  0.8679  0.8758  0.8726  0.8732  0.0048  
MAE (testset)     0.6697  0.6765  0.6686  0.6715  0.6695  0.6712  0.0028  
Fit time          1.18    1.22    1.63    1.27    1.20    1.30    0.17    
Test time         0.11    0.10    0.26    0.11    0.10    0.14    0.06    


{'test_rmse': array([0.86875695, 0.88092589, 0.86790862, 0.87578182, 0.87258328]),
 'test_mae': array([0.66974362, 0.67645239, 0.66859186, 0.67147973, 0.6695167 ]),
 'fit_time': (1.184931993484497,
  1.216374158859253,
  1.6340382099151611,
  1.2699167728424072,
  1.1979939937591553),
 'test_time': (0.10705375671386719,
  0.10466742515563965,
  0.25809264183044434,
  0.11285281181335449,
  0.10258364677429199)}

- cross_valdate(): 폴드별 성능 평가 수치, 전체 폴드의 평균 성능 평가 수치 보여줌

- Surprise의 GridSearchCV: 교차 검증을 통한 하이퍼 파라미터 최적화 수행

SVD: 점진적 하강 방식의 반복 횟수 지정 n_epochs, SVD의 잠재 요인 K의 크기 지정 n_factors를 튜닝

In [45]:
# n_epochs: [20, 40, 60], 'n_factors': [50, 100, 200]으로 변경, CV=3일 때 최적 하이퍼 파라미터 도출
# tratings.csv를 DataFrame으로 로딩한 데이터 그대로 사용

from surprise.model_selection import GridSearchCV

# 최적화할 파라미터를 딕셔너리 형태로 지정.
param_grid={'n_epochs': [20, 40, 60], 'n_factors': [50, 100, 200] }

# CV를 3개 폴드 세트로 지정, 성능 평가는 rmse, mse로 수행하도록 GridSearchCV 구성
gs=GridSearchCV(SVD, param_grid, measures=['rmse', 'mae',], cv=3)
gs.fit(data)

# 최고 RMSE Evaluation 점수와 그때의 하이퍼 파라미터
print(gs.best_score['rmse'])
print(gs.best_params['rmse'])

0.8776388808022743
{'n_epochs': 20, 'n_factors': 50}


# Surprise를 이용한 개인화 영화 추천 시스템 구축

Surprise를 이용하여 잠재 요인 협업 필터링 기반의 개인화된 영화 추천 구현

<앞 예제>

학습 데이터로 fit() 호출 -> 테스트 데잍로 test() 호출 -> 예측 평점 계산, MSE/RMSE로 성능 평가

<뒷 과정>

Surprise 패키지로 학습된 추천 알고리즘 기반으로, 특정 사용자가 아직 평점을 매기지 않은 영화 중 개인 취향에 가장 적절한 영화 추천

<예제>

- ratings.csv 전체를 학습 데이터로 사용 (데이터를 학습 데이터 / 테스트 데이터로 분리 X)

- Surprise는 데이터 세트를 train_test_split()을 이용하여 내부에서 사용하는 Trainset 클래스 객체로 변환해주어야 함.

In [48]:
# 다음 코드는 train_test_split()으로 분리되지 않는 데이터 세트에 fit()을 호출해 오류가 발생합니다.
data=Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']], reader)
algo=SVD(n_factors=50, random_state=0)
algo.fit(data)

AttributeError: 'DatasetAutoFolds' object has no attribute 'n_users'

- 데이터 세트 전체를 학습 데이터로 사용하려면, DatasetAutoFolds 클래스 이용

- DatasetAutoFolds 객체 생성 후, build_full_trainset() 메서드 호출 -> 전체 데이터가 학습 데이터 세트

In [50]:
from surprise.dataset import DatasetAutoFolds

reader=Reader(line_format='user item rating timestamp', sep=',', rating_scale=(0.5,5))
# DatasetAutoFolds 클래스를 ratings_noh.csv 파일 기반으로 생성.
data_folds=DatasetAutoFolds(ratings_file='ratings_noh.csv', reader=reader)

# 전체 데이터를 학습 데이터로 생성함.
trainset=data_folds.build_full_trainset()

In [51]:
# 특정 사용자에게 영화를 추천하기 위해, 아직 보지 않은 영화 목록 확인
# SVD를 이용한 학습 수행

algo=SVD(n_epochs=20, n_factors=50, random_state=0)
algo.fit(trainset)

- 특정 사용자는 userId=9인 사용자로 지정

- userId 9가 아직 평점을 매기지 않은 영화를 movieId 42로 선정하여 예측 평점 계산

- 영화의 상세 정보는 movies.csv 파일에 있으므로, 해당 파일을 DataFrame으로 로딩

In [53]:
# 영화에 대한 상세 속성 정보 DataFrame 로딩
movies=pd.read_csv('movies.csv')

# userId=9의 movieId를 추출해 movieId=42 데이터가 있는지 확인.
movieIds=ratings[ratings['userId']==9]['movieId']

if movieIds[movieIds==42].count()==0:
  print('사용자 아이디 9는 영화 아이디 42의 평점 없음')

print(movies[movies['movieId']==42])

사용자 아이디 9는 영화 아이디 42의 평점 없음
    movieId                   title              genres
38       42  Dead Presidents (1995)  Action|Crime|Drama


- 학습된 SVD 객체에서 predict() 메서드 내의 iserId, movieId 값 입력 (모두 문자열 값)

In [54]:
uid=str(9)
iid=str(42)

pred=algo.predict(uid, iid, verbose=True)

user: 9          item: 42         r_ui = None   est = 3.13   {'was_impossible': False}


- 추천 예측 평점(est)=3.13

- 앞 예제: 사용자가 평점을 매기지 않은 영화의 추천 예측 평점 구하기

- 이번 예제: 사용자가 평점을 매기지 않은 전체 영화를 추출한 뒤 예측 평점 순으로 영화 추천

1. 추천 대상이 되는 영화 추출

2. Surprise 내부의 데이터 객체에 대한 액세스 제약 등으로 인해 get_unseen_movies() 사용 X, get_unseen_surprise() 함수 생성

-> 아이디 9인 사용자가 평점을 매기지 않은 영화 정보 반환

In [56]:
def get_unseen_surprise(ratings, movies, userId):
  # 입력값으로 들어온 userId에 해당하는 사용자가 평점을 매긴 모든 영화를 리스트로 생성
  seen_movies=ratings[ratings['userId']==userId]['movieId'].tolist()

  # 모든 영화의 movieId를 리스트로 생성.
  total_movies=movies['movieId'].tolist()

  # 모든 영화의 movieId 중 이미 평점을 매긴 영화의 movieId를 제외한 후 리스트로 생성
  unseen_movies=[movie for movie in total_movies if movie not in seen_movies]
  print('평점 매긴 영화 수:', len(seen_movies), '추천 대상 영화 수:', len(unseen_movies),
  '전체 영화 수:', len(total_movies))

  return unseen_movies

  unseen_movies=get_unseen_surprise(ratings, movies, 9)


- 사용자 아이디 9번은 전체 9742개 영화 중 46개만 평점을 매김.

- 추천 대상 영화는 9696개

(이 중 학습된 추천 알고리즘 클래스인 SVD를 이용하여 높은 예측 평점을 가진 순으로 영화 추천)

recomm_movie_by_surprise() 함수

- 인자: 학습이 완료된 추천 알고리즘 객체, 추천 대상 사용자 아이디, 추천 대상 영화 리스트 객체, 추천 상위 N개 개수)

- 추천 대상 영화 모두를 대상으로, 추천 알고리즘 객체의 predict() 메서드 호출, 결과인 Prediction 객체를 리스트 객체로 저장

- 저장된 리스트 내부의 Prediction 객체를 예측 평점이 높은 순으로 정렬

- Top-N개의 Prediction 객체에서 영화 아이디, 영화 제목, 예측 평점 정보 추출 반환


In [60]:
def recomm_movie_by_surprise(algo, userId, unseen_movies, top_n=10):

  # 알고리즘 객체의 predict() 메서드를 평점이 없는 영화에 반복 수행한 후 결과를 list 객체로 저장
  predictions=[algo.predict(str(userId), str(movieId)) for movieId in unseen_movies]

  #  predictions list 객체는 surprise의 Predictions 객체를 원소로 가지고 있음.
  # [Prediction(uid='9', iid='1', est=3.69), Prediction(uid='9', iid='2', est=2.98),,,,]

  # 이를 est 값으로 정렬하기 위해서 아래의 sortkey_est 함수를 정의함.
  # sortkey_est 함수는 list 객체의 sort() 함수의 키 값으로 사용되어 정렬 수행.
  def sortkey_est(pred):
    return pred.est

  # sortkey_est() 반환값의 내림 차순으로 정렬 수행하고 top_n개의 최상위 값 추출.
  predictions.sort(key=sortkey_est, reverse=True)
  top_predictions=predictions[:top_n]

  # top_n으로 추출된 영화의 정보 추출. 영화 아이디, 추천 예상 평점, 제목 추출
  top_movie_ids=[int(pred.iid) for pred in top_predictions]
  top_movie_rating=[pred.est for pred in top_predictions]
  top_movie_titles=movies[movies.movieId.isin(top_movie_ids)]['title']

  top_movie_preds=[(id, title, rating) for id, title, rating in \
                     zip(top_movie_ids, top_movie_titles, top_movie_rating)]

  return top_movie_preds

unseen_movies=get_unseen_surprise(ratings, movies, 9)
top_movie_preds=recomm_movie_by_surprise(algo, 9, unseen_movies, top_n=10)

print('##### Top-10 추천 영화 리스트 #####')
for top_movie in top_movie_preds:
  print(top_movie[1], ":", top_movie[2])

평점 매긴 영화 수: 46 추천 대상 영화 수: 9696 전체 영화 수: 9742
##### Top-10 추천 영화 리스트 #####
Usual Suspects, The (1995) : 4.306302135700814
Star Wars: Episode IV - A New Hope (1977) : 4.281663842987387
Pulp Fiction (1994) : 4.278152632122759
Silence of the Lambs, The (1991) : 4.226073566460876
Godfather, The (1972) : 4.1918097904381995
Streetcar Named Desire, A (1951) : 4.154746591122657
Star Wars: Episode V - The Empire Strikes Back (1980) : 4.122016128534504
Star Wars: Episode VI - Return of the Jedi (1983) : 4.108009609093436
Goodfellas (1990) : 4.083464936588478
Glory (1989) : 4.07887165526957


- 9번 아이디 사용자에게 서스펜스/스릴러/범죄 영화 및 액션 영화(케빈 스페이시 주연의 '유주얼 서스펙트', '펄프픽션', '양들의 침묵', '대부', '좋은 아이들') 추천

- Surprise 패키지는 복잡한 알고리즘을 직접 구현하지 않고도 쉽고 간결한 API를 이용해 파이썬 기반에서 추천 시스템 구축 가능

# 정리

추천 시스템의 대표 방식: 콘텐츠 기반 필터링, 협업 필터링

- 콘텐츠 기반 필터링

: 아이템을 구성하는 여러 가지 콘텐츠 중 사용자가 좋아하는 콘텐츠를 필터링하여 이에 맞는 아이템 추천 방식

(장르, 감독, 영화배우, 영화 키워드와 같은 요소들)

: 요소들을 결합하여 하나의 콘텐츠 특징으로 피처 벡터화 진행

-> 가장 유사한 다른 피처 벡터를 가진 영화 추천

- 협업 필터링

1. 최근접 이웃 협업 필터링

- 사용자 기반 / 아이템 기반

(아이템 기반(유사도 기준: 사용자들의 아이템에 대한 평가를 벡터화한 값) 추천 방식)

모든 아이템을 행으로, 모든 사용자를 열로,

아이템-사용자 행렬 데이터 세트 생성 (사용자별- 아이템 평점을 값으로 하는)

아이템별 코사인 유사도를 이용한 사요자 평점 피처 벡터에 따른 Top-N 유사 아이템 추천

2. 잠재 요인 협업 필터링

- 사용자-아이템 평점 행렬 데이터에 숨어 있는 잠재 요인을 추출하여 사용자가 아직 평점을 매기지 않은 아이템에 대한 평점을 예측하여 이를 추천에 반영하는 방식

행렬 분해: 다차원의 사용자-아이템 평점 행렬을 저차원의 사용자-잠재요인, 아이템-잠재요인 행렬로 분해

- Surprise

간단한 API만을 이요하여 파이썬 기반 추천 시스템 구현 (사이킷런과 유사한 API)